In [2]:
import numpy as np

In [3]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

tf.config.optimizer.set_jit(True)
mixed_precision.set_global_policy('mixed_float16')

print("Num GPUs Available:", len(gpus))
print("Mixed Precision Policy:", mixed_precision.global_policy())

Num GPUs Available: 1
Mixed Precision Policy: <DTypePolicy "mixed_float16">


In [4]:
from tensorflow.keras.models import load_model
model = load_model('outputs/cnn-model-50epochs-fast-99_84.h5')

I0000 00:00:1775552011.334419    3740 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [5]:
class_name = ['glinoma', 'meningioma', 'notumor', 'pituitary']

In [7]:
from tensorflow.keras.preprocessing import image

In [34]:
image_path = "datasets/brain-tumor-mri-dataset/Testing/notumor/Te-no_18.jpg"

In [35]:
img_array = image.img_to_array(image.load_img(image_path, target_size=(224, 224)))/255.0
img_array

array([[[0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255],
        ...,
        [0.02745098, 0.02745098, 0.02745098],
        [0.02745098, 0.02745098, 0.02745098],
        [0.02745098, 0.02745098, 0.02745098]],

       [[0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255],
        ...,
        [0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255]],

       [[0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255],
        [0.03137255, 0.03137255, 0.03137255],
        ...,
        [0.03529412, 0.03529412, 0.03529412],
        [0.03529412, 0.03529412, 0.03529412],
        [0.03529412, 0.03529412, 0.03529412]],

       ...,

       [[0.03529412, 0.03529412, 0.03529412],
        [0.03529412, 0.03529412, 0.03529412],
        [0.03529412, 0

In [36]:
img = np.expand_dims(img_array, axis=0)
img.shape

(1, 224, 224, 3)

In [37]:
prediction = model.predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


In [38]:
print("Predicted Class:", class_name[np.argmax(prediction)])

Predicted Class: notumor


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array, load_img

def test_random_images(model=None,
                       test_dir='datasets/brain-tumor-mri-dataset/Testing',
                       n=20,
                       image_size=(224,224),
                       show=False):
    test_path = Path(test_dir)
    if not test_path.exists():
        raise FileNotFoundError(f"Test directory not found: {test_dir}")
    class_names = sorted([p.name for p in test_path.iterdir() if p.is_dir()])
    print("Found classes:", class_names)
    exts = ('.jpg', '.jpeg', '.png', '.bmp')
    img_paths = [p for p in test_path.rglob('*') if p.suffix.lower() in exts and p.is_file()]
    if len(img_paths) == 0:
        raise FileNotFoundError("No image files found under the test directory.")
    print(f"Found {len(img_paths)} images in testing folders.")

    # sample images
    n = min(n, len(img_paths))
    sampled = random.sample(img_paths, n)

    if model is None:
        if 'model' in globals() and globals()['model'] is not None:
            model = globals()['model']
            print("Using `model` from notebook globals.")
        else:
            fallback = Path('outputs/model_50epochs_fast.keras')
            if fallback.exists():
                model = tf.keras.models.load_model(str(fallback))
                print(f"Loaded model from {fallback}")
            else:
                raise ValueError("No model provided, no `model` in globals, and fallback file not found.")

    correct = 0
    wrong = 0
    details = []

    images_to_show = []
    titles = []

    for p in sampled:
        true_label = p.parent.name
        img = load_img(str(p), target_size=image_size)
        arr = img_to_array(img).astype('float32') / 255.0
        batch = np.expand_dims(arr, axis=0)
        preds = model.predict(batch, verbose=0)
        pred_idx = int(np.argmax(preds, axis=1)[0])
        pred_label = class_names[pred_idx] if pred_idx < len(class_names) else str(pred_idx)
        is_correct = (pred_label == true_label)
        if is_correct:
            correct += 1
        else:
            wrong += 1
        details.append({'path': str(p), 'true': true_label, 'pred': pred_label, 'correct': is_correct})

        if show:
            images_to_show.append(arr)
            titles.append(f"T:{true_label} | P:{pred_label} {'✓' if is_correct else '✗'}")

    accuracy = correct / n if n > 0 else 0.0
    print(f"Tested {n} images — Correct: {correct}, Wrong: {wrong}, Accuracy: {accuracy:.4f}")

    if show and images_to_show:
        cols = 4
        rows = int(np.ceil(len(images_to_show) / cols))
        fig, axs = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        axs = np.array(axs).reshape(-1)
        for i, ax in enumerate(axs):
            if i < len(images_to_show):
                ax.imshow(images_to_show[i])
                ax.set_title(titles[i])
                ax.axis('off')
            else:
                ax.axis('off')
        plt.tight_layout()
        plt.show()

    return {'total': n, 'correct': correct, 'wrong': wrong, 'accuracy': accuracy, 'details': details}

In [41]:
summary = test_random_images(model=None, test_dir='datasets/brain-tumor-mri-dataset/Testing', n=100, show=False)
print(summary)

Found classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Found 1600 images in testing folders.
Using `model` from notebook globals.
Tested 100 images — Correct: 92, Wrong: 8, Accuracy: 0.9200
{'total': 100, 'correct': 92, 'wrong': 8, 'accuracy': 0.92, 'details': [{'path': 'datasets/brain-tumor-mri-dataset/Testing/notumor/Te-no_330.jpg', 'true': 'notumor', 'pred': 'notumor', 'correct': True}, {'path': 'datasets/brain-tumor-mri-dataset/Testing/meningioma/Te-aug-me_73.jpg', 'true': 'meningioma', 'pred': 'meningioma', 'correct': True}, {'path': 'datasets/brain-tumor-mri-dataset/Testing/notumor/Te-no_135.jpg', 'true': 'notumor', 'pred': 'notumor', 'correct': True}, {'path': 'datasets/brain-tumor-mri-dataset/Testing/notumor/Te-no_351.jpg', 'true': 'notumor', 'pred': 'notumor', 'correct': True}, {'path': 'datasets/brain-tumor-mri-dataset/Testing/meningioma/Te-me_236.jpg', 'true': 'meningioma', 'pred': 'meningioma', 'correct': True}, {'path': 'datasets/brain-tumor-mri-dataset/Testing/gl